# 00 — Start here: data, progress, and reproducibility

This is the project entrance. **All six notebooks live in `lava-aws-multilingual-docvqa/notebooks/`.** Read 00 → 01 → 02 → 03 → 04 → 05. Their saved outputs are included, so viewing them requires no GPU or login. Each notebook can also run independently after the environment is installed.

The data download, full PDF audit, three reader pilots, and first full-document retrieval experiment are complete. Notebook 05 connects retrieval and the reader and shows whether the final integrated pilot has been measured. Kaggle test inference and submission are optional extensions, not prerequisites for this scoped portfolio. These notebooks explain completed work; they do not launch model experiments.

## 1. Verify the data already acquired

Training contains every available labeled example: 16 questions from five PDFs. Test contains 624 questions from 200 PDFs, with answers hidden. The sample submission is a template, not ground truth. We inspect the saved audit below; this cell does not repeat the download.

In [1]:
import json
from pathlib import Path

from IPython.display import HTML, display

from lava.evaluation.walkthrough import TABLE_STYLE, render_table
from lava.notebook_support import find_repo_root
from lava.readers.runtime_logging import RuntimeEventLogger

ROOT = find_repo_root(Path.cwd())
logger = RuntimeEventLogger("notebook.protocol")
with logger.stage("01_verify_data", heartbeat_seconds=15):
    manifest = json.loads((ROOT / "reports/raw_data_manifest_summary.json").read_text())
    audit = json.loads((ROOT / "reports/data_audit/data_audit_summary_full.json").read_text())
    assert manifest["complete"] and manifest["verified_file_count"] == 208
    assert audit["audit_mode"] == "full" and audit["audited_pdf_count_for_mode"] == 205
    rows = []
    for profile in audit["csv_profiles"]:
        if profile["selected_columns"]["question"] is None:
            continue
        split = "Training" if profile["selected_columns"]["answer"] else "Test"
        rows.append(
            {
                "Split": split,
                "Questions": profile["row_count"],
                "PDFs": profile["referenced_document_count"],
                "Japanese": profile["language_counts"]["ja"],
                "Vietnamese": profile["language_counts"]["vi"],
            }
        )
    display(HTML(TABLE_STYLE + render_table(rows, caption="Data audit complete")))
    display(
        HTML(
            render_table(
                [
                    {"Check": "Verified raw files", "Result": manifest["verified_file_count"]},
                    {"Check": "PDFs audited", "Result": audit["audited_pdf_count_for_mode"]},
                    {
                        "Check": "Exact PDF duplicates across splits",
                        "Result": audit["cross_split_exact_duplicate_pdf_group_count"],
                    },
                ],
                caption="Checks completed before modeling",
            )
        )
    )

{"component": "notebook.protocol", "elapsed_seconds": 0.0, "event": "01_verify_data.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T01:38:10.835+00:00"}


Split,Questions,PDFs,Japanese,Vietnamese
Test,624,200,587,37
Training,16,5,15,1


Check,Result
Verified raw files,208
PDFs audited,205
Exact PDF duplicates across splits,0


{"component": "notebook.protocol", "elapsed_seconds": 0.003, "event": "01_verify_data.completed", "level": "INFO", "stage_elapsed_seconds": 0.003, "timestamp_utc": "2026-09-07T01:38:10.838+00:00"}


## 2. Understand the evaluation boundary

LAVA combines semantic answer credit and evidence-page F1, with equal weight, for each question. Our local judge uses a pinned Gemma checkpoint and validated prompt. The organizer's exact judging runtime is not public, so our local score is not a leaderboard result.

Five training PDFs are too few for a precise generalization claim. Inspect document-level results alongside question averages. Any future tuning must respect document isolation; these frozen pilot comparisons are descriptive and do not claim that nested cross-validation has already been run.

In [2]:
with logger.stage("02_inspect_protocol", heartbeat_seconds=15):
    protocol = json.loads((ROOT / "configs/evaluation_protocol.lock.json").read_text())
    constraints = protocol["competition_constraints"]
    display(
        HTML(
            render_table(
                [
                    {
                        "Contract": "Local primary score",
                        "Value": "Mean of semantic answer credit and evidence-page F1",
                    },
                    {
                        "Contract": "Independent evaluation unit",
                        "Value": "PDF; report all five document effects",
                    },
                    {"Contract": "Random seed", "Value": protocol["random_seed"]},
                    {
                        "Contract": "Recorded runtime ceiling (seconds)",
                        "Value": constraints["maximum_end_to_end_inference_seconds"],
                    },
                    {
                        "Contract": "Recorded single-GPU memory ceiling (GiB)",
                        "Value": constraints["maximum_single_gpu_vram_gib"],
                    },
                    {
                        "Contract": "Final inference package",
                        "Value": "Docker; complete runtime compliance still to verify",
                    },
                ],
                caption="Frozen evaluation and submission contract",
            )
        )
    )

{"component": "notebook.protocol", "elapsed_seconds": 0.008, "event": "02_inspect_protocol.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T01:38:10.842+00:00"}


Contract,Value
Local primary score,Mean of semantic answer credit and evidence-page F1
Independent evaluation unit,PDF; report all five document effects
Random seed,20260902
Recorded runtime ceiling (seconds),7200
Recorded single-GPU memory ceiling (GiB),40
Final inference package,Docker; complete runtime compliance still to verify


{"component": "notebook.protocol", "elapsed_seconds": 0.009, "event": "02_inspect_protocol.completed", "level": "INFO", "stage_elapsed_seconds": 0.001, "timestamp_utc": "2026-09-07T01:38:10.843+00:00"}


## 3. Know what is finished and what is next

**9B is the provisional reader.** The 4B, 9B, and 27B comparisons are complete on all 16 training questions. There is no need to repeat them to read these results. Notebook 03 explains their quality, cost, and limitations; Notebook 04 measures evidence retrieval.

The final portfolio milestone is the fixed 9B retrieved-page pilot in Notebook 05, followed by publication of its actual results, limitations, and passing quality checks. The implementation is available through `make finish CHARGES=YES`; the notebook explicitly distinguishes a pending GPU run from measured results. Full test inference and Kaggle submission remain optional, separately validated extensions.

In [3]:
with logger.stage("03_locate_current_work", heartbeat_seconds=15):
    display(
        HTML(
            render_table(
                [
                    {
                        "Notebook": "00",
                        "Purpose": "Data and progress",
                        "State": "Data acquired; full audit complete",
                    },
                    {
                        "Notebook": "01",
                        "Purpose": "Experiment design",
                        "State": "Protocol and reader configurations recorded",
                    },
                    {
                        "Notebook": "02",
                        "Purpose": "Execution evidence",
                        "State": "Three complete 16-question reader pilots",
                    },
                    {
                        "Notebook": "03",
                        "Purpose": "Model quality and cost",
                        "State": "4B, 9B, 27B scored; provisional 9B",
                    },
                    {
                        "Notebook": "04",
                        "Purpose": "Evidence retrieval",
                        "State": "74 pages evaluated; 9B integration next",
                    },
                ],
                caption="Your notebook sequence",
            )
        )
    )
logger.emit("protocol.walkthrough.completed", raw_files=manifest["verified_file_count"])

{"component": "notebook.protocol", "elapsed_seconds": 0.013, "event": "03_locate_current_work.started", "level": "INFO", "stage_elapsed_seconds": 0.0, "timestamp_utc": "2026-09-07T01:38:10.848+00:00"}


Notebook,Purpose,State
00,Data and progress,Data acquired; full audit complete
01,Experiment design,Protocol and reader configurations recorded
02,Execution evidence,Three complete 16-question reader pilots
03,Model quality and cost,"4B, 9B, 27B scored; provisional 9B"
04,Evidence retrieval,74 pages evaluated; 9B integration next


{"component": "notebook.protocol", "elapsed_seconds": 0.014, "event": "03_locate_current_work.completed", "level": "INFO", "stage_elapsed_seconds": 0.001, "timestamp_utc": "2026-09-07T01:38:10.849+00:00"}


{"component": "notebook.protocol", "elapsed_seconds": 0.015, "event": "protocol.walkthrough.completed", "level": "INFO", "raw_files": 208, "timestamp_utc": "2026-09-07T01:38:10.849+00:00"}


## 4. Use one workspace

Open `notebooks/01_oracle_reader_benchmark_design.ipynb` next. In SageMaker, the project is `/home/sagemaker-user/lava-aws-multilingual-docvqa`.

- `notebooks/` contains the six canonical, executed notebooks.
- `src/`, `scripts/`, `configs/`, and `tests/` contain the implementation and checks.
- `reports/` contains measured results and notebook checksum manifests.
- `artifacts/` contains ignored runtime caches and logs. You do not need to navigate into it to read the project.

`make notebooks` verifies and reuses current notebook outputs, or refreshes only notebooks whose inputs changed. `make quality` runs the explicit test and code-quality gate. Long GPU runs execute as independent SageMaker jobs; they can outlive a browser session. CPU analysis stops if Studio stops, but completed model and retrieval checkpoints persist in S3.

[Continue to Notebook 01](01_oracle_reader_benchmark_design.ipynb) · [Project overview](../README.md)